# Chapter 4 · Embeddings and semantic search

Turn sentences into vectors, measure their similarity, build an FAQ search engine that works by meaning and compare it with one that works by words, chunk a long document and draw a map of meanings.

This notebook does not use any language model, only an embedding model, so it **works fine without a GPU**. With a GPU it takes less than a minute; without one, a few minutes. If you want the GPU anyway: Runtime → Change runtime type → T4 GPU → Save.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. The model and the data

The first time, it downloads the embedding model (a little over 1 GB), the FAQ and the terms and conditions.

In [ ]:
"""Chapter 4: embeddings and semantic search. Works on a GPU or a CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"


def load_embeddings(name=EMBEDDING_MODEL):
    """The embedding model, on the GPU if there is one."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Model {name} on {device}")
    return SentenceTransformer(name, device=device)


def download(name):
    """A file from the data folder; on Colab, it is downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return path.read_text(encoding="utf-8")

In [ ]:
model = load_embeddings()
data = json.loads(download("faq.json"))
document = download("terms_and_conditions.md")
print(len(data["questions"]), "FAQ questions,", len(data["queries"]), "test queries")
print(len(document.split()), "words in the terms and conditions")

## 2. Embeddings and similarities

In [ ]:
def embed(texts, kind="passage"):
    """Normalized embeddings. E5 wants "query:" before questions and "passage:"
    before the texts being searched."""
    return model.encode([f"{kind}: {t}" for t in texts],
                        normalize_embeddings=True, convert_to_numpy=True)


def similarities(sentences):
    """Matrix of cosine similarities between some sentences."""
    vectors = embed(sentences, "query")
    return vectors @ vectors.T

In [ ]:
v = embed(["How long does shipping take?"], "query")
print(v.shape, "length:", np.linalg.norm(v[0]))
print(v[0][:8])

In [ ]:
SENTENCES = ["How long does shipping take?",
             "How many days until my package gets here?",
             "¿Cuánto tarda el envío?",
             "What warranty does the battery have?",
             "I love riding my bike up mountain passes."]

In [ ]:
for sentence, row in zip(SENTENCES, similarities(SENTENCES)):
    print(f"{sentence[:42]:42s}", "  ".join(f"{s:.2f}" for s in row))

## 3. Search by meaning and by words

In [ ]:
class SemanticSearch:
    """A vector index: a matrix with one embedding per row."""

    def __init__(self, texts):
        """Embeds the texts and keeps them as the rows of the index."""
        self.texts = texts
        self.matrix = embed(texts, "passage")

    def search(self, query, k=3):
        """The k most similar texts, as (position, similarity) pairs."""
        scores = self.matrix @ embed([query], "query")[0]
        best = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in best]

In [ ]:
STOP_WORDS = set(
    "a an the and or but of to in on at for from by with about as is are was be "
    "am been do does did i me my mine you your yours we us our it its this that "
    "these those there here what how when where which who why can could will "
    "would should if not no so any some have has had get got".split())


def strip_accents(text):
    """Lowercase and without accents."""
    return "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")


class WordSearch:
    """Classic word search with TF-IDF: it counts shared words, giving more
    weight to those that appear in few texts."""

    def __init__(self, texts):
        """Learns the vocabulary and the weights from the texts."""
        self.texts = texts
        stop = sorted({strip_accents(w) for w in STOP_WORDS})
        self.tfidf = TfidfVectorizer(preprocessor=strip_accents, stop_words=stop)
        self.matrix = self.tfidf.fit_transform(texts)

    def search(self, query, k=3):
        """The k texts with the highest TF-IDF score, as (position, score) pairs."""
        scores = (self.matrix @ self.tfidf.transform([query]).T).toarray()[:, 0]
        best = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in best]

In [ ]:
texts = [q["question"] + " " + q["answer"] for q in data["questions"]]
semantic, words = SemanticSearch(texts), WordSearch(texts)
query = "Where can I see where my package is?"
for name, searcher in (("Meaning", semantic), ("Words", words)):
    print(name)
    for i, score in searcher.search(query):
        print(f"   {score:.2f}  {data['questions'][i]['question']}")

In [ ]:
def evaluate_faq(searcher, queries):
    """Hits in first position and among the top three."""
    at_1 = at_3 = 0
    for query, correct in queries:
        positions = [i + 1 for i, _ in searcher.search(query, k=3)]
        at_1 += positions[0] == correct
        at_3 += correct in positions
    return at_1, at_3


def test_faq(data):
    """FAQ search: by meaning versus by words."""
    texts = [q["question"] + " " + q["answer"] for q in data["questions"]]
    results = {}
    for name, cls in (("words", WordSearch),
                      ("meaning", SemanticSearch)):
        at_1, at_3 = evaluate_faq(cls(texts), data["queries"])
        results[name] = (at_1, at_3)
        print(f"By {name:7s}: first position {at_1}/30, "
              f"among the top three {at_3}/30")
    return results

In [ ]:
faq = test_faq(data)

## 4. Chunk a long document

In [ ]:
def sections(document):
    """One chunk per section of the document (those that start with ##)."""
    return [s.strip() for s in re.split(r"\n(?=## )", document) if s.startswith("##")]


def chunk(document, words, overlap=0):
    """Chunks with a fixed number of words; each one repeats the last
    "overlap" words of the previous one."""
    text = re.sub(r"#+ ", "", document).split()
    step = words - overlap
    return [" ".join(text[i:i + words])
            for i in range(0, max(len(text) - overlap, 1), step)]

In [ ]:
chunks = chunk(document, 40)
print(len(chunks), "chunks of 40 words. The first two:")
print(chunks[0], "\n---\n", chunks[1])

In [ ]:
TERMS_QUESTIONS = [
    ("Can I cancel an order I've already paid for?", "can be canceled at no cost"),
    ("If I pay by bank transfer, how long do I have?", "must arrive within five days"),
    ("Do you keep my card numbers?", "never stores card details"),
    ("What happens if I'm not home when the delivery driver comes?",
     "will try a second time"),
    ("The box arrived dented, what do I need to do?",
     "note it on the carrier's delivery receipt"),
    ("Can I return bib shorts I've already opened?",
     "returns of bib shorts or technical underwear"),
    ("Will you also refund what I paid for shipping?",
     "original shipping costs are only refunded"),
    ("Can I exchange shoes I bought online at the store?",
     "we do not make direct exchanges in the physical store"),
    ("What capacity does the battery have to keep under warranty?",
     "at least 70%"),
    ("What does the $35 tune-up include?", "adjusting the gears and brakes"),
    ("How do I ask you to delete my data?", "privacy@northcycles.com"),
    ("How long do you take to answer emails?", "within 48 business hours"),
    ("If a product sells out after I order it, what happens?",
     "we will let the customer know within 24 hours"),
    ("Do your prices include sales tax?", "does not charge sales tax"),
    ("From Hawaii, who pays for return shipping?",
     "from alaska and hawaii, the cost of return shipping"),
]


def evaluate_chunking(chunks, questions=TERMS_QUESTIONS):
    """Does the retrieved chunk contain the sentence that answers the question?"""
    searcher = SemanticSearch(chunks)
    at_1 = at_3 = 0
    for question, sentence in questions:
        retrieved = [chunks[i].lower() for i, _ in searcher.search(question, 3)]
        at_1 += sentence in retrieved[0]
        at_3 += any(sentence in c for c in retrieved)
    return at_1, at_3


def test_chunkings(document):
    """Different ways of chunking the terms and conditions."""
    variants = {"By sections": sections(document)}
    for words in (40, 80, 160, 320):
        for overlap in (0, words // 4):
            variants[f"{words} words, overlap {overlap}"] = chunk(
                document, words, overlap)
    results = {}
    for name, chunks in variants.items():
        at_1, at_3 = evaluate_chunking(chunks)
        results[name] = (len(chunks), at_1, at_3)
        print(f"{name:26s} {len(chunks):3d} chunks  "
              f"first {at_1:2d}/15  top three {at_3:2d}/15")
    return results

In [ ]:
chunkings = test_chunkings(document)

## 5. A map of meanings

In [ ]:
def faq_map(data):
    """2-D coordinates of the FAQ questions, with PCA."""
    vectors = embed([q["question"] for q in data["questions"]], "query")
    return PCA(n_components=2).fit_transform(vectors)

In [ ]:
import matplotlib.pyplot as plt

xy = faq_map(data)
topics = sorted({q["topic"] for q in data["questions"]})
plt.figure(figsize=(8, 5.5))
for topic in topics:
    points = [(x, y, q["id"]) for (x, y), q in zip(xy, data["questions"]) if q["topic"] == topic]
    plt.scatter([p[0] for p in points], [p[1] for p in points], label=topic)
    for x, y, i in points:
        plt.annotate(str(i), (x, y), xytext=(4, 3), textcoords="offset points", fontsize=7)
plt.legend()
plt.show()

## Exercises

1. Explain in your own words what a sentence embedding is and why two sentences with no words in common can have close embeddings.

2. Compute by hand the cosine similarity between the vectors (1, 2, 2) and (2, 0, 1). Normalize both vectors and check that their dot product gives the same result.

3. Add to `SENTENCES` a sentence that says the opposite of the first one, such as "Shipping is instant, it takes no time at all." What is its similarity with "How long does shipping take?"? What does that tell you about what embeddings capture?

4. Write ten queries that use the same words as the FAQ questions and repeat the comparison between the two search engines. Which one wins now?

5. Write three queries with an order number or a specific model, such as "Has NC-10482 shipped yet?", and add to the FAQ a text that contains it. Which search engine finds it?

6. Remove the `query:` and `passage:` prefixes from `embed`. Do the hits change?

7. Change `EMBEDDING_MODEL` to `intfloat/multilingual-e5-small` and repeat everything. How many hits do you lose, and how much time do you gain?

8. Combine sections and fixed size: chunk by sections and, if a section has more than 120 words, split it into chunks of 80 words with an overlap of 20. Compare with the table.

9. Replace PCA with t-SNE (`sklearn.manifold.TSNE`, with `perplexity=5`) and draw the map again. Are the topics better separated? What drawbacks does t-SNE have?

10. **Interactive search.** Write a function that takes a query and shows the three most similar FAQ questions with their answer and their similarity. Try it with ten queries of your own and note which ones it gets wrong. Do you see a pattern?

## References

Model [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base), by Microsoft, under the MIT license. [Sentence-BERT](https://arxiv.org/abs/1908.10084) (Reimers and Gurevych, 2019), [E5](https://arxiv.org/abs/2212.03533) (Wang et al., 2022) and [MTEB](https://arxiv.org/abs/2210.07316) (Muennighoff et al., 2022). The North Cycles store, its FAQ and its terms and conditions are fictitious. Code and experiments are the author's own.